# Лабораторная работа №6
## Реализация модели SIR в подходе сетей Петри

В данном скрипте выполняется базовое моделирование эпидемиологической
модели SIR, представленной в виде сети Петри.

Рассматриваются два варианта моделирования:

- детерминированная симуляция на основе системы ОДУ;
- стохастическая симуляция с использованием алгоритма Гиллеспи.

В результате работы скрипта сохраняются таблицы с результатами
моделирования и графики динамики состояний `S`, `I`, `R`.

In [ ]:
using DrWatson
@quickactivate "project"

## Подключение библиотек

Для воспроизводимости стохастического эксперимента используется модуль
`Random`. Основная логика модели вынесена в файл `src/SIRPetri.jl`.

In [ ]:
using Random

include(srcdir("SIRPetri.jl"))
using .SIRPetri

using DataFrames, CSV, Plots

## Параметры модели

В модели используются два основных параметра:

- `β` — коэффициент заражения;
- `γ` — коэффициент выздоровления.

Также задаётся максимальное время моделирования `tmax`.

In [ ]:
β = 0.3
γ = 0.1
tmax = 100.0

## Построение сети Петри

Функция `build_sir_network` создаёт сеть Петри для модели SIR.
В этой сети позиции соответствуют состояниям:

- `S` — восприимчивые;
- `I` — инфицированные;
- `R` — выздоровевшие.

Переход `infection` описывает заражение, а переход `recovery`
описывает выздоровление.

In [ ]:
net, u0, states = build_sir_network(β, γ)

## Детерминированная симуляция

Сначала выполняется детерминированное моделирование.
В этом случае динамика системы описывается системой обыкновенных
дифференциальных уравнений.

Результат сохраняется в таблицу `data/sir_det.csv`.

In [ ]:
df_det = simulate_deterministic(
    net,
    u0,
    (0.0, tmax),
    saveat = 0.5,
    rates = [β, γ],
)

CSV.write(datadir("sir_det.csv"), df_det)

## Стохастическая симуляция

Далее выполняется стохастическая симуляция.
В отличие от детерминированного случая, здесь события заражения
и выздоровления происходят случайным образом.

Для воспроизводимости результата фиксируется зерно генератора
случайных чисел.

In [ ]:
Random.seed!(123)

df_stoch = simulate_stochastic(
    net,
    u0,
    (0.0, tmax),
    rates = [β, γ],
)

CSV.write(datadir("sir_stoch.csv"), df_stoch)

## Построение графиков

Для обеих симуляций строятся графики изменения численности групп
`S`, `I`, `R` во времени.

Детерминированный график сохраняется в файл
`plots/sir_det_dynamics.png`.

In [ ]:
p_det = plot_sir(df_det)
savefig(p_det, plotsdir("sir_det_dynamics.png"))

Стохастический график сохраняется в файл
`plots/sir_stoch_dynamics.png`.

In [ ]:
p_stoch = plot_sir(df_stoch)
savefig(p_stoch, plotsdir("sir_stoch_dynamics.png"))

## Результат

После выполнения скрипта в каталоге `data/` появляются таблицы:

- `sir_det.csv`;
- `sir_stoch.csv`.

В каталоге `plots/` появляются графики:

- `sir_det_dynamics.png`;
- `sir_stoch_dynamics.png`.

Эти результаты используются для анализа различий между
детерминированным и стохастическим подходами к моделированию SIR.

In [ ]:
println("Базовый прогон завершён. Результаты в data/ и plots/")